# Cell communication analysis: mcTECs

In [ ]:
import os
import sys
from datetime import datetime
today = datetime.today().strftime('%Y-%m-%d')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import anndata as ad
import hdf5plugin
#import liana as li

import warnings
warnings.filterwarnings('ignore', category=ad.ImplicitModificationWarning)

# Add repo path to sys path (allows to access scripts and metadata from repo)
repo_path,_ = os.path.split(os.path.split(os.getcwd())[0])
repo_path = '/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/General_analysis'
sys.path.insert(1, repo_path) 
sys.path.insert(2, '/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/General_analysis/scripts')

# Autoreload custom scripts
%load_ext autoreload
%autoreload 2

# Define paths
plots_path = f'{repo_path}/plots/'
data_path = f'{repo_path}/data/'
model_path = os.path.join(repo_path, 'models')
general_data_path = '/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/General_analysis/data'

print('Dir for plots: {}'.format(plots_path))
print('Dir for data: {}'.format(data_path))

# Formatting
from matplotlib import font_manager
font_manager.fontManager.addfont("/nfs/team205/ny1/ThymusSpatialAtlas/software/Arial.ttf")
plt.style.use('/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/General_analysis/scripts/plotting/thyAgeing.mplstyle')

# Import custom scripts
from utils import get_latest_version,update_obs,freq_by_donor
from anno_levels import get_ct_levels, get_ct_palette, age_group_levels, age_group_palette
from plotting.utils import plot_grouped_boxplot, calc_figsize

In [ ]:
# Define columns
col_cell_type_broad = 'taa_l3'
col_cell_type_fine = 'taa_l4'
# col_cell_type_broad_levels = get_ct_levels(col_cell_type_broad, taa_l1 = ['T', 'NK'])
# col_cell_type_fine_levels = get_ct_levels(col_cell_type_fine, taa_l1 = ['T', 'NK'])
col_age_group = 'age_group'
col_age_group_levels = eval(f'{col_age_group}_levels')

In [ ]:
ct_anno = pd.read_csv(f'{general_data_path}/objects/rna/thyAgeing_all_scvi_v5_2025-10-15_curatedAnno_v11.csv', index_col = 0)

recirc_barcodes = ct_anno[ct_anno['taa_l2'].isin(['T_recirc', 'B_cell', 'B_plasma', 'Mac', 'Neutrophil'])].index.tolist()

# Load adata
object_version = 'v5_2025-04-03'
adata = ad.read_h5ad(f'{general_data_path}/objects/rna/thyAgeing_all_scvi_{object_version}.zarr', backed = 'r')

# Subset adata to only include mcTEC cells
adata_recirc = adata[recirc_barcodes, :].to_memory()

# Add annotations to adata
adata_recirc.obs = adata_recirc.obs.join(ct_anno, how = 'left')
adata_recirc = adata_recirc[(adata_recirc.obs['anno_status'] == 'include') & (adata_recirc.obs['qc_status'] == 'PASS')]

# Update metadata
latest_meta_path = get_latest_version(dir = f'{general_data_path}/metadata', file_prefix='Thymus_ageing_metadata')
latest_meta = pd.read_excel(latest_meta_path)
update_obs(adata_recirc, latest_meta, on = 'index', ignore_warning = True)

adata_recirc.shape

Filter genes to receptors/ligands expressed in recirc:

In [ ]:
# Load CellPhoneDB database (v5.0.0)
ia_db = pd.read_csv(f'{data_path}/references/ccc_databases/cellphoneDB_v5.0.0_interaction_input.csv')
ia_db[['ligand', 'receptor']] = ia_db['interactors'].str.rsplit('-', n=1, expand=True)
ia_db['receptor'] = ia_db['receptor'].str.split('+')
ia_db = ia_db.explode('receptor')
ia_db.head()

In [ ]:
# Calculate fraction of cells expressing each ligand/receptor in each cell type
adata_recirc = adata_recirc[:, adata_recirc.var_names.isin(ia_db['ligand'].unique().tolist() + ia_db['receptor'].unique().tolist())]
recirc_dummies = pd.get_dummies(adata_recirc.obs['taa_l2'])

recirc_frac = (recirc_dummies.values.T @ (adata_recirc.X > 0).astype(int)) / recirc_dummies.sum(axis = 0).values[:, None]
recirc_frac.shape

# Plot fraction expressed by cell type
recirc_frac_df = pd.DataFrame(
    recirc_frac,
    index=recirc_dummies.columns,
    columns=adata_recirc.var_names
).reset_index(names='cell_type')

recirc_frac_long = recirc_frac_df.melt(id_vars='cell_type', var_name='gene', value_name='frac_expressed')

plt.figure(figsize=calc_figsize(width=90, height=60))
sns.histplot(
    data=recirc_frac_long,
    x='frac_expressed',
    hue='cell_type',
    bins=50,
    element='step',
    stat='density',
    common_norm=False
)
plt.xlabel('Fraction expressed')
plt.ylabel('Density')
plt.yscale('log')
plt.title('Fraction expressed by cell type in adata_recirc')
plt.tight_layout()
plt.show()

In [ ]:
lr_oi = adata_recirc.var_names[(recirc_frac >= 0.1).sum(axis=0) > 0].tolist()
print(f'Number of ligands/receptors expressed in >10% of cells for any given cell type: {len(lr_oi)}')

ia_oi = ia_db[(ia_db['receptor'].isin(lr_oi)) & (ia_db['ligand'].isin(adata.var_names))]
print(f'Number of interactions with receptor expressed in >10% of cells: {ia_oi.shape[0]}')

all_lr = set(ia_oi['ligand'].unique().tolist() + ia_oi['receptor'].unique().tolist())
print(f'Number of unique ligands/receptors corresponding to these interactions: {len(all_lr)}')

Identify cell types which could communicate with mcTEc based on niche assignments:

In [ ]:
niche_comp = pd.read_csv('/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/Spatial_analyses/data/morpho_niches2/inflammatoryNiche/recirc_niche_k9_taa_l3_comb_composition_pct.csv', index_col=0)
niche_recovery = pd.read_csv('/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/Spatial_analyses/data/morpho_niches2/inflammatoryNiche/recirc_niche_k9_per_donor_pct.csv', index_col=0)
niche_recovery['age_group'] = ['adult' if x.startswith('A') else 'paed' if x.startswith('Z') else 'infant' for x in niche_recovery.index]
niche_recovery

In [ ]:
niche_recovery.groupby('age_group').mean().T.plot(kind='bar', figsize=calc_figsize(width=80, height=60))

Paed-dominant:
- N3: deep medulla (T + Bmem)
- N7: medulla + vessel (T+B+Mac)

Adult-dominant:
- N1: deep medulla (T+Bmem)
- N6: medulla (T+B) -> higher proportion of infiltrating cells than N7
- N2: vasc+adipo+fibro (B plasma,B,Cd4)
- N4: vasc+adipo+fibro (Mac)
- N8: cortex (+Mac)

Shared:
- N5: septal
- N0: CMJ

## Inflamed medulla

In [ ]:
pct_thres = 10
medulla_niches = {'adult' : ['recirc_N1', 'recirc_N6'],
                  'paed' : ['recirc_N3', 'recirc_N7'],}

niche_ratios = niche_recovery.loc[:, [col for col in niche_recovery.columns if any(niche in col for niche in medulla_niches['adult'] + medulla_niches['paed'])] + ['age_group']]
niche_ratios = niche_ratios.groupby('age_group').mean().T
niche_ratios = niche_ratios.div(niche_ratios.sum(axis=0), axis=1)

niche_comp_combined = niche_comp.loc[[col for col in niche_comp.index if any(niche in col for niche in medulla_niches['adult'] + medulla_niches['paed'])],:]
niche_comp_combined, niche_ratios

In [ ]:
niche_comp_combined = pd.DataFrame((niche_ratios.values.T @ niche_comp_combined.values) / 100, index=niche_ratios.columns, columns=niche_comp_combined.columns)
niche_comp_combined

In [ ]:
niche_comp_combined.loc[:,niche_comp_combined.columns.str.startswith('EC')]

In [ ]:
ct_prop_threshold = 0.01

ct_per_niche = {
    a: niche_comp_combined.columns[niche_comp_combined.loc[a] >= ct_prop_threshold].tolist()
    for a in niche_comp_combined.index
}
all_ct_oi = set([ct for ct_list in ct_per_niche.values() for ct in ct_list])

import pprint
pprint.pprint(ct_per_niche, compact=True, width=120)
pprint.pprint(all_ct_oi, compact=True, width=120)


In [ ]:
[c for c in all_ct_oi if c not in ct_anno['taa_l3'].values]

In [ ]:
# Subset adata
adata_sub_barcodes = ct_anno[ct_anno['taa_l3'].isin(all_ct_oi)].index.tolist()
adata_sub = adata[adata_sub_barcodes,adata.var_names.isin(all_lr)].to_memory()

# Add annotations to adata
adata_sub.obs = adata_sub.obs.join(ct_anno, how = 'left')
adata_sub = adata_sub[(adata_sub.obs['anno_status'] == 'include') & (adata_sub.obs['qc_status'] == 'PASS')]

# Update metadata
latest_meta_path = get_latest_version(dir = f'{general_data_path}/metadata', file_prefix='Thymus_ageing_metadata')
latest_meta = pd.read_excel(latest_meta_path)
update_obs(adata_sub, latest_meta, on = 'index', ignore_warning = True)

adata_sub.obs['cell_type'] = adata_sub.obs['taa_l3']

adata_sub.obs['cell_type'].value_counts()

Inlcude donors with 100 recirc cells

In [ ]:
# Filter out donors with fewer than 50 mcTECs
min_cells = 100
col_sample = 'donor'
ncells_by_sample = adata_sub[adata_sub.obs['taa_l2'].isin(['T_recirc', 'B_cell', 'B_plasma', 'Mac', 'Neutrophil'])].obs[col_sample].value_counts().to_frame(name = 'n_cells')
print('Removing {} donors with fewer than {} cells'.format(ncells_by_sample[ncells_by_sample['n_cells'] < min_cells].shape[0], min_cells))
adata_sub = adata_sub[~adata_sub.obs[col_sample].isin(ncells_by_sample[ncells_by_sample['n_cells'] < min_cells].index)]

In [ ]:
# Level to cell type assignment
ct_levels = [c for c in get_ct_levels('taa_l3') if c in all_ct_oi]
import pprint
pprint.pprint(ct_levels, compact = True)

In [ ]:
# Niche composition for each cell type by age group
ct_freq = niche_comp_combined.melt(ignore_index = False, var_name='cell_type', value_name='niche_prop').reset_index()
ct_freq = ct_freq.loc[ct_freq['cell_type'].isin(ct_levels), :]
ct_freq

In [ ]:
interactions_oi = ia_oi[['ligand', 'receptor']].drop_duplicates().apply(lambda x: (x['ligand'], x['receptor']), axis = 1).tolist()
interactions_oi 

In [ ]:
# Calculate fraction of cells expressing each ligand/receptor in each cell type and donor
adata_sub.obs['ct_donor'] = adata_sub.obs['donor'] + '__' + adata_sub.obs['cell_type']
ct_by_donor_dummy = pd.get_dummies(adata_sub.obs['ct_donor'])
n_cells_per_ct_donor = ct_by_donor_dummy.sum(axis=0)
X_bool = (adata_sub.X > 0).astype(int)

frac_by_ct = ct_by_donor_dummy.values.T * X_bool
frac_by_ct = frac_by_ct / n_cells_per_ct_donor.to_numpy()[:, None]

frac_by_ct

In [ ]:
frac_by_ct_df = pd.DataFrame(frac_by_ct, index=ct_by_donor_dummy.columns, columns=adata_sub.var_names)
frac_by_ct_df['cell_type'] = frac_by_ct_df.index.str.split('__').str[-1]
frac_by_ct_df['donor'] = frac_by_ct_df.index.str.split('__').str[0]
frac_by_ct_df = frac_by_ct_df.melt(id_vars=['cell_type', 'donor'], var_name='gene', value_name='frac_expressed')
frac_by_ct_df.head()

In [ ]:
lr_frac_expressed = ia_oi[['ligand', 'receptor']].drop_duplicates() \
    .merge(frac_by_ct_df, left_on='ligand', right_on='gene', how='left').rename(columns={'frac_expressed': 'ligand_props', 'cell_type' : 'source'}).drop(columns = 'gene') \
    .merge(frac_by_ct_df, left_on=['receptor', 'donor'], right_on=['gene', 'donor'], how='left').rename(columns={'frac_expressed': 'receptor_props', 'cell_type': 'target'}).drop(columns = 'gene') 
lr_frac_expressed['age_group'] = lr_frac_expressed['donor'].map(adata_sub.obs[['donor', 'age_group']].drop_duplicates().set_index('donor')['age_group'])
lr_frac_expressed['interaction'] = lr_frac_expressed['ligand'] + '->' + lr_frac_expressed['receptor']
lr_frac_expressed['cells'] = lr_frac_expressed['source'] + '->' + lr_frac_expressed['target']
lr_frac_expressed = lr_frac_expressed.merge(ct_freq[['age_group', 'cell_type', 'niche_prop']], left_on=['age_group', 'source'], right_on=['age_group', 'cell_type'], how='left').rename(columns={'niche_prop': 'source_prop'}).drop(columns=['cell_type']) \
    .merge(ct_freq[['age_group', 'cell_type','niche_prop']], left_on=['age_group', 'target'], right_on=['age_group', 'cell_type'], how='left').rename(columns={'niche_prop': 'target_prop'}).drop(columns=['cell_type'])
#lr_frac_expressed['age_group'] = np.where(lr_frac_expressed['age_group'] == 'adult', 'adult', 'paed') # Combine paed and infant into one group for comparison with of paed shared niche and adult-specific niche

lr_frac_expressed

In [ ]:
# Subset to interactions with mcTEC as target
ct_recirc_l3 = adata_sub[adata_sub.obs['taa_l2'].isin(['T_recirc', 'B_cell', 'B_plasma', 'Mac', 'Neutrophil'])].obs['taa_l3'].unique()
lr_frac_expressed = lr_frac_expressed[lr_frac_expressed['target'].isin(ct_recirc_l3)].dropna()
lr_frac_expressed.shape

In [ ]:
lr_frac_expressed.to_csv(f'{data_path}/analyses/cellComm/recirc/combined_niche_lr_expressed_by_interaction_and_celltype.csv', index=False)

Replace source/target proportion with niche-specific proportions

In [ ]:
lr_frac_expressed = pd.read_csv(f'{data_path}/analyses/cellComm/recirc/combined_niche_lr_expressed_by_interaction_and_celltype.csv')
lr_frac_expressed

In [ ]:
# Calculate mean properties (using all samples, not just TOT sort)
lr_frac_expressed['tot_prob'] = lr_frac_expressed['ligand_props'] * lr_frac_expressed['receptor_props'] * lr_frac_expressed['source_prop'] * lr_frac_expressed['target_prop']
mean_props = lr_frac_expressed.groupby(['interaction', 'cells', 'age_group']).agg({'ligand_props' : 'mean', 'receptor_props' : 'mean', 'tot_prob': 'mean'}).reset_index()
mean_props = mean_props.pivot(index = ['interaction', 'cells'], columns = 'age_group', values = ['ligand_props', 'receptor_props','tot_prob']).reset_index()
mean_props.columns = ['_'.join(col).strip() for col in mean_props.columns.values]
mean_props = mean_props.rename(columns = {'interaction_': 'interaction', 'cells_': 'cells'})
mean_props.head()

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from scipy.stats import ranksums
from statsmodels.stats.multitest import multipletests

# Filter interactions
# Select interactions with at least one age group with expression > 0.1
target_age_group = ['adult', 'paed']
min_expr = 0.1

# Keeping any interaction with at least one age group with expression > in expr
#ia_to_keep = mean_props.loc[mean_props[['ligand_props_' + ag for ag in target_age_group] + ['receptor_props_' + ag for ag in target_age_group]].ge(min_expr).all(axis=1)][['interaction', 'cells']]

# Keeping interactions where L and R are both expressed at min expr in the same age group
ia_to_keep = []
for g in target_age_group:
    ia_to_keep.append(mean_props.loc[mean_props[[f'ligand_props_{g}', f'receptor_props_{g}']].ge(min_expr).all(axis=1)][['interaction', 'cells']])
ia_to_keep = pd.concat(ia_to_keep, axis = 0).drop_duplicates()

print(f'Number of interactions to keep: {ia_to_keep.shape[0]}')

donor_n = lr_frac_expressed[['donor', col_age_group]].drop_duplicates().groupby('age_group').agg({'donor': 'count'}).to_dict()['donor']
# Define a function to process a single cell
def process_cells(ct):
    ias = ia_to_keep[ia_to_keep['cells'] == ct]['interaction'].unique()
    test_ranksum = []
    for ia in ias:
        test_data = lr_frac_expressed[(lr_frac_expressed['interaction'] == ia) & (lr_frac_expressed['cells'] == ct)][['donor', col_age_group, 'tot_prob']].dropna(subset=['tot_prob']).copy()
        group1 = test_data[test_data['age_group'] == target_age_group[0]]['tot_prob'].to_numpy()
        if group1.shape[0] < donor_n[target_age_group[0]]:
            group1 = np.concatenate((group1, np.repeat(0, repeats=donor_n[target_age_group[0]] - group1.shape[0])))
        group2 = test_data[test_data['age_group'] == target_age_group[1]]['tot_prob'].to_numpy()
        if group2.shape[0] < donor_n[target_age_group[1]]:
            group2 = np.concatenate((group2, np.repeat(0, repeats=donor_n[target_age_group[1]] - group2.shape[0])))
        test_res = ranksums(group1, group2)
        test_ranksum.append([ia, ct, test_res[0], test_res[1]])
    test_ranksum = pd.DataFrame(test_ranksum, columns=['interaction', 'cells', 'statistic', 'pvalue'])
    test_ranksum['padj'] = multipletests(test_ranksum['pvalue'], method='fdr_bh')[1]
    return test_ranksum

# Use ThreadPoolExecutor with 4 threads to process cells in parallel
with ThreadPoolExecutor(max_workers=4) as executor:
    results = list(executor.map(process_cells, ia_to_keep['cells'].unique()))

# Combine results
ranksums_res = pd.concat(results, axis=0)
ranksums_res.to_csv(f'{data_path}/analyses/cellComm/recirc/combined_niche_{target_age_group[1]}_vs_{target_age_group[0]}_ranksums.csv')

ranksums_res.sort_values('statistic', ascending=False)

In [ ]:
ranksums_res = pd.read_csv(f'{data_path}/analyses/cellComm/recirc/combined_niche_{target_age_group[1]}_vs_{target_age_group[0]}_ranksums.csv', index_col = 0)
ranksums_res[['source', 'target']] = ranksums_res['cells'].str.split('->', expand=True)
ranksums_res[['ligand', 'receptor']] = ranksums_res['interaction'].str.split('->', expand=True)
ranksums_res.sort_values('statistic', ascending=False)

In [ ]:
ranksums_res_filtered = ranksums_res[ranksums_res['padj'] < 0.05]
ranksums_res_filtered['ia_change_direction'] = np.where(ranksums_res_filtered['statistic'] > 0, 'up', 'down')
ranksums_res_filtered.sort_values('statistic', ascending=False)

In [ ]:
ranksums_res_filtered['source'].value_counts()

In [ ]:
# Adult niche vs combined paed+infant niche
pprint.pprint(ranksums_res_filtered.groupby(['cells','ia_change_direction'])['interaction'].agg(list).to_dict(), compact=True, width=120)

In [ ]:
# Adult niche vs infant niche
import pprint
pprint.pprint(ranksums_res_filtered.groupby(['cells','ia_change_direction'])['interaction'].agg(list).to_dict(), compact=True, width=120)

In [ ]:
# Paed niche vs infant niche
import pprint
pprint.pprint(ranksums_res_filtered.groupby(['cells','ia_change_direction'])['interaction'].agg(list).to_dict(), compact=True, width=120)

In [ ]:
# Adult niche vs paediatric niche
import pprint
pprint.pprint(ranksums_res_filtered.groupby(['cells','ia_change_direction'])['interaction'].agg(list).to_dict(), compact=True, width=120)

## Changes in non-inflamed medulla

In [ ]:
pct_thres = 10
medulla_niches = {'adult' : ['recirc_N1', 'recirc_N6'],
                  'paed' : ['recirc_N3', 'recirc_N7'],}

niche_ratios = niche_recovery.loc[:, [col for col in niche_recovery.columns if any(niche in col for niche in medulla_niches['paed'])] + ['age_group']]
niche_ratios = niche_ratios.groupby('age_group').mean().T
niche_ratios = niche_ratios.div(niche_ratios.sum(axis=0), axis=1)

niche_comp_shared = niche_comp.loc[[col for col in niche_comp.index if any(niche in col for niche in medulla_niches['paed'])],:]

niche_comp_shared = pd.DataFrame((niche_ratios.values.T @ niche_comp_shared.values) / 100, index=niche_ratios.columns, columns=niche_comp_shared.columns)
niche_comp_shared

In [ ]:
ct_prop_threshold = 0.02

ct_per_niche = {
    a: niche_comp_shared.columns[niche_comp_shared.loc[a] >= ct_prop_threshold].tolist()
    for a in niche_comp_shared.index
}
all_ct_oi = set([ct for ct_list in ct_per_niche.values() for ct in ct_list])

import pprint
pprint.pprint(ct_per_niche, compact=True, width=120)
pprint.pprint(all_ct_oi, compact=True, width=120)

In [ ]:
# Subset adata
adata_sub_barcodes = ct_anno[ct_anno['taa_l3'].isin(all_ct_oi)].index.tolist()
adata_sub = adata[adata_sub_barcodes,adata.var_names.isin(all_lr)].to_memory()

# Add annotations to adata
adata_sub.obs = adata_sub.obs.join(ct_anno, how = 'left')
adata_sub = adata_sub[(adata_sub.obs['anno_status'] == 'include') & (adata_sub.obs['qc_status'] == 'PASS')]

# Update metadata
latest_meta_path = get_latest_version(dir = f'{general_data_path}/metadata', file_prefix='Thymus_ageing_metadata')
latest_meta = pd.read_excel(latest_meta_path)
update_obs(adata_sub, latest_meta, on = 'index', ignore_warning = True)

adata_sub.obs['cell_type'] = adata_sub.obs['taa_l3']

adata_sub.obs['cell_type'].value_counts()

In [ ]:
# Filter out donors with fewer than 100 recirculating cells (T_recirc, B_cell, B_plasma, Mac, Neutrophil)
min_cells = 100
col_sample = 'donor'
ncells_by_sample = adata_sub[adata_sub.obs['taa_l2'].isin(['T_recirc', 'B_cell', 'B_plasma', 'Mac', 'Neutrophil'])].obs[col_sample].value_counts().to_frame(name = 'n_cells')
print('Removing {} donors with fewer than {} cells'.format(ncells_by_sample[ncells_by_sample['n_cells'] < min_cells].shape[0], min_cells))
adata_sub = adata_sub[~adata_sub.obs[col_sample].isin(ncells_by_sample[ncells_by_sample['n_cells'] < min_cells].index)]

In [ ]:
# Level to cell type assignment
ct_levels = [c for c in get_ct_levels('taa_l3') if c in all_ct_oi]
import pprint
pprint.pprint(ct_levels, compact = True)

In [ ]:
# Niche composition for each cell type by age group
ct_freq = niche_comp_shared.melt(ignore_index = False, var_name='cell_type', value_name='niche_prop').reset_index()
ct_freq = ct_freq.loc[ct_freq['cell_type'].isin(ct_levels), :]
ct_freq

In [ ]:
# Calculate fraction of cells expressing each ligand/receptor in each cell type and donor
adata_sub.obs['ct_donor'] = adata_sub.obs['donor'] + '__' + adata_sub.obs['cell_type']
ct_by_donor_dummy = pd.get_dummies(adata_sub.obs['ct_donor'])
n_cells_per_ct_donor = ct_by_donor_dummy.sum(axis=0)
X_bool = (adata_sub.X > 0).astype(int)

frac_by_ct = ct_by_donor_dummy.values.T * X_bool
frac_by_ct = frac_by_ct / n_cells_per_ct_donor.to_numpy()[:, None]

frac_by_ct

In [ ]:
frac_by_ct_df = pd.DataFrame(frac_by_ct, index=ct_by_donor_dummy.columns, columns=adata_sub.var_names)
frac_by_ct_df['cell_type'] = frac_by_ct_df.index.str.split('__').str[-1]
frac_by_ct_df['donor'] = frac_by_ct_df.index.str.split('__').str[0]
frac_by_ct_df = frac_by_ct_df.melt(id_vars=['cell_type', 'donor'], var_name='gene', value_name='frac_expressed')
frac_by_ct_df.head()

In [ ]:
lr_frac_expressed = ia_oi[['ligand', 'receptor']].drop_duplicates() \
    .merge(frac_by_ct_df, left_on='ligand', right_on='gene', how='left').rename(columns={'frac_expressed': 'ligand_props', 'cell_type' : 'source'}).drop(columns = 'gene') \
    .merge(frac_by_ct_df, left_on=['receptor', 'donor'], right_on=['gene', 'donor'], how='left').rename(columns={'frac_expressed': 'receptor_props', 'cell_type': 'target'}).drop(columns = 'gene') 
lr_frac_expressed['age_group'] = lr_frac_expressed['donor'].map(adata_sub.obs[['donor', 'age_group']].drop_duplicates().set_index('donor')['age_group'])
lr_frac_expressed['interaction'] = lr_frac_expressed['ligand'] + '->' + lr_frac_expressed['receptor']
lr_frac_expressed['cells'] = lr_frac_expressed['source'] + '->' + lr_frac_expressed['target']
lr_frac_expressed = lr_frac_expressed.merge(ct_freq[['age_group', 'cell_type', 'niche_prop']], left_on=['age_group', 'source'], right_on=['age_group', 'cell_type'], how='left').rename(columns={'niche_prop': 'source_prop'}).drop(columns=['cell_type']) \
    .merge(ct_freq[['age_group', 'cell_type','niche_prop']], left_on=['age_group', 'target'], right_on=['age_group', 'cell_type'], how='left').rename(columns={'niche_prop': 'target_prop'}).drop(columns=['cell_type'])
#lr_frac_expressed['age_group'] = np.where(lr_frac_expressed['age_group'] == 'adult', 'adult', 'paed') # Combine paed and infant into one group for comparison with of paed shared niche and adult-specific niche

lr_frac_expressed

In [ ]:
# Subset to interactions with mcTEC as target
ct_recirc_l3 = adata_sub[adata_sub.obs['taa_l2'].isin(['T_recirc', 'B_cell', 'B_plasma', 'Mac', 'Neutrophil'])].obs['taa_l3'].unique()
lr_frac_expressed = lr_frac_expressed[lr_frac_expressed['target'].isin(ct_recirc_l3)].dropna()
lr_frac_expressed.shape

In [ ]:
lr_frac_expressed.to_csv(f'{data_path}/analyses/cellComm/recirc/shared_niche_lr_expressed_by_interaction_and_celltype.csv', index=False)

In [ ]:
# Calculate mean properties (using all samples, not just TOT sort)
lr_frac_expressed['tot_prob'] = lr_frac_expressed['ligand_props'] * lr_frac_expressed['receptor_props'] * lr_frac_expressed['source_prop'] * lr_frac_expressed['target_prop']
mean_props = lr_frac_expressed.groupby(['interaction', 'cells', 'age_group']).agg({'ligand_props' : 'mean', 'receptor_props' : 'mean', 'tot_prob': 'mean'}).reset_index()
mean_props = mean_props.pivot(index = ['interaction', 'cells'], columns = 'age_group', values = ['ligand_props', 'receptor_props','tot_prob']).reset_index()
mean_props.columns = ['_'.join(col).strip() for col in mean_props.columns.values]
mean_props = mean_props.rename(columns = {'interaction_': 'interaction', 'cells_': 'cells'})
mean_props.head()

In [ ]:
lr_frac_expressed[['donor', 'age_group']].drop_duplicates().groupby('age_group').agg({'donor': 'count'}).to_dict()['donor']

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from scipy.stats import ranksums
from statsmodels.stats.multitest import multipletests

# Filter interactions
# Select interactions with at least one age group with expression > 0.1
target_age_group = ['paed', 'infant']
min_expr = 0.1

# Keeping any interaction with at least one age group with expression > in expr
#ia_to_keep = mean_props.loc[mean_props[['ligand_props_' + ag for ag in target_age_group] + ['receptor_props_' + ag for ag in target_age_group]].ge(min_expr).all(axis=1)][['interaction', 'cells']]

# Keeping interactions where L and R are both expressed at min expr in the same age group
ia_to_keep = []
for g in target_age_group:
    ia_to_keep.append(mean_props.loc[mean_props[[f'ligand_props_{g}', f'receptor_props_{g}']].ge(min_expr).all(axis=1)][['interaction', 'cells']])
ia_to_keep = pd.concat(ia_to_keep, axis = 0).drop_duplicates()

print(f'Number of interactions to keep: {ia_to_keep.shape[0]}')

donor_n = adata.obs[['donor', col_age_group]].drop_duplicates().groupby('age_group').agg({'donor': 'count'}).to_dict()['donor']
# Define a function to process a single cell
def process_cells(ct):
    ias = ia_to_keep[ia_to_keep['cells'] == ct]['interaction'].unique()
    test_ranksum = []
    for ia in ias:
        test_data = lr_frac_expressed[(lr_frac_expressed['interaction'] == ia) & (lr_frac_expressed['cells'] == ct)][['donor', col_age_group, 'tot_prob']].dropna(subset=['tot_prob']).copy()
        group1 = test_data[test_data['age_group'] == target_age_group[0]]['tot_prob'].to_numpy()
        if group1.shape[0] < donor_n[target_age_group[0]]:
            group1 = np.concatenate((group1, np.repeat(0, repeats=donor_n[target_age_group[0]] - group1.shape[0])))
        group2 = test_data[test_data['age_group'] == target_age_group[1]]['tot_prob'].to_numpy()
        if group2.shape[0] < donor_n[target_age_group[1]]:
            group2 = np.concatenate((group2, np.repeat(0, repeats=donor_n[target_age_group[1]] - group2.shape[0])))
        test_res = ranksums(group1, group2)
        test_ranksum.append([ia, ct, test_res[0], test_res[1]])
    test_ranksum = pd.DataFrame(test_ranksum, columns=['interaction', 'cells', 'statistic', 'pvalue'])
    test_ranksum['padj'] = multipletests(test_ranksum['pvalue'], method='fdr_bh')[1]
    return test_ranksum

# Use ThreadPoolExecutor with 4 threads to process cells in parallel
with ThreadPoolExecutor(max_workers=4) as executor:
    results = list(executor.map(process_cells, ia_to_keep['cells'].unique()))

# Combine results
ranksums_res = pd.concat(results, axis=0)
ranksums_res.to_csv(f'{data_path}/analyses/cellComm/recirc/shared_niche_{target_age_group[1]}_vs_{target_age_group[0]}_ranksums.csv')

ranksums_res.sort_values('statistic', ascending=False)

In [ ]:
ranksums_res = pd.read_csv(f'{data_path}/analyses/cellComm/recirc/shared_niche_{target_age_group[1]}_vs_{target_age_group[0]}_ranksums.csv', index_col = 0)
ranksums_res[['source', 'target']] = ranksums_res['cells'].str.split('->', expand=True)
ranksums_res[['ligand', 'receptor']] = ranksums_res['interaction'].str.split('->', expand=True)
ranksums_res.sort_values('statistic', ascending=False)

In [ ]:
ranksums_res_filtered = ranksums_res[ranksums_res['padj'] < 0.05]
ranksums_res_filtered['ia_change_direction'] = np.where(ranksums_res_filtered['statistic'] > 0, 'up', 'down')
ranksums_res_filtered.sort_values('statistic', ascending=False)

In [ ]:
# Adult niche vs infant niche
pprint.pprint(ranksums_res_filtered.groupby(['cells','ia_change_direction'])['interaction'].agg(list).to_dict(), compact=True, width=120)

In [ ]:
# Adult niche vs paed niche
pprint.pprint(ranksums_res_filtered.groupby(['cells','ia_change_direction'])['interaction'].agg(list).to_dict(), compact=True, width=120)

In [ ]:
# paed niche vs infant niche
pprint.pprint(ranksums_res_filtered.groupby(['cells','ia_change_direction'])['interaction'].agg(list).to_dict(), compact=True, width=120)

In [ ]:
import pandas as pd
ranksums_res = pd.read_csv(f'/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/General_analysis/data/analyses/cellComm/recirc/shared_niche_paed_vs_adult_ranksums.csv', index_col = 0)

ranksums_res

In [ ]:
ranksums_res.loc[ranksums_res['interaction'].str.startswith('CCL2')]

## Plotting

In [ ]:
MODULES = {
    
    'Recruitment' : {
    # ---------------------------------------------------------------- #
    # i. RECRUITMENT                                                    #
    # ---------------------------------------------------------------- #

    "Recruitment: EC tethering and rolling": [
        "CD34->SELL",
        "PODXL->SELL",
        "PODXL2->SELL",
        "SELE->GLG1",
        "SELPLG->SELL",
    ],

    "Recruitment: EC arrest chemokines": [
        "CXCL12->CXCR4",
        "CCL14->CCR1",
        "CCL23->CCR1",
        "CCL2->CCR2",
    ],

    "Recruitment: EC firm adhesion": [
        "VCAM1->ITGA4",
        "VCAM1->ITGB1",
        "VCAM1->ITGB2",
        "VCAM1->ITGB7",
        "VCAM1->ITGA9",
        "ICAM1->ITGAL",
        "ICAM1->ITGB2",
        "ICAM1->SPN",
        "ICAM2->ITGAL",
        "ICAM2->ITGB2",
    ],

    "Recruitment: EC junctional and diapedesis": [
        "JAM2->ITGA4",
        "JAM2->ITGB1",
        "F11R->ITGAL",
        "F11R->ITGB2",
        "PECAM1->CD38",
    ],

    "Recruitment: fibroblast lymphoid-organiser chemokines": [
        "CCL19->CCR7",
        "CCL21->CCR7",
        "CCL19->CCRL2",
        "CXCL12->CXCR4",
        "CXCL14->CXCR4",
        "CCL2->CCR2",
        "CCL8->CCR2",
    ],

    "Recruitment: epithelial chemoattraction": [
        "CCL19->CCR7",
        "CCL21->CCR7",
        "CXCL10->CXCR3",
        "CXCL11->CXCR3",
        "CXCL12->CXCR4",
        "CXCL14->CXCR4",
        "CCL20->CCR6",
    ],

    "Recruitment: plasma-cell homing": [
        "CCL28->CCR10",
    ],
    
    },

    # ---------------------------------------------------------------- #
    # ii. RETENTION                                                     #
    # ---------------------------------------------------------------- #
    
    "Retention" : {

    "Retention: epithelial basement membrane": [
        "COL4A1->ITGB1",
        "COL4A2->ITGB1",
        "COL4A3->ITGB1",
        "COL4A4->ITGB1",
        "COL4A5->ITGB1",
        "COL4A6->ITGB1",
        "COL7A1->ITGB1",
        "COL17A1->ITGB1",
        "LAMA3->ITGB1",
        "LAMC1->ITGB1",
        "LAMC1->ITGA6",
        "LAMC1->ITGAV",
    ],

    "Retention: fibroblast fibrillar collagen": [
        "COL1A1->ITGB1",
        "COL1A2->ITGB1",
        "COL3A1->ITGB1",
        "COL5A1->ITGB1",
        "COL5A2->ITGB1",
        "COL5A3->ITGB1",
        "COL27A1->ITGB1",
        "COL6A1->ITGB1",
        "COL6A2->ITGB1",
        "COL6A3->ITGB1",
        "COL6A6->ITGB1",
        "FBN1->ITGB1",
        "FBN1->ITGA5",
    ],

    "Retention: fibril-associated and network collagens": [
        "COL12A1->ITGB1",
        "COL14A1->ITGB1",
        "COL16A1->ITGB1",
        "COL8A1->ITGB1",
        "COL8A2->ITGB1",
        "COL9A2->ITGB1",
        "COL22A1->ITGB1",
        "COL15A1->ITGB1",
        "COL18A1->ITGB1",
    ],

    "Retention: fibronectin, tenascin and thrombospondin": [
        "FN1->ITGA4",
        "FN1->ITGA5",
        "FN1->ITGAV",
        "FN1->ITGB1",
        "FN1->ITGB5",
        "FN1->ITGB7",
        "TNC->ITGA4",
        "TNC->ITGA9",
        "TNC->ITGAV",
        "TNC->ITGB1",
        "THBS1->ITGB1",
        "THBS1->ITGAV",
        "THBS1->CD36",
        "THBS2->CD36",
    ],

    "Retention: residency programming": [
        "CDH1->ITGAE",
        "CDH1->ITGB1",
        "CDH1->ITGB7",
    ],

    "Retention: adhesion and co-stimulatory contact": [
        "ALCAM->CD6",
        "CD58->CD2",
        "ICAM1->ITGAL",
        "F11R->ITGAL",
        "JAM3->ITGB2",
        "JAM3->ITGAM",
    ],

    "Retention: TGF-beta (permissive for ITGAE)": [
        "TGFB1->TGFBR1",
        "TGFB1->TGFBR2",
        "TGFB1->ITGAV",
        "TGFB3->TGFBR1",
        "TGFB3->TGFBR2",
        "TGFB2->TGFBR1",
        "TGFB2->TGFBR2",
    ],
    
    },

    # ---------------------------------------------------------------- #
    # iii. STATE AND SURVIVAL                                           #
    # ---------------------------------------------------------------- #
    
    'State and survival' : {

    "Survival: B-lineage BAFF/APRIL": [
        "TNFSF13B->TNFRSF13B",
        "TNFSF13B->TNFRSF13C",
        "TNFSF13->TNFRSF13B",
        "TNFSF13->TNFRSF17",
    ],

    "Survival: B-lineage gp130": [
        "IL6->IL6R",
        "IL6->IL6ST",
        "CLCF1->IL6ST",
        "OSM->IL6ST",
    ],

    "Survival: B-lineage IL-10": [
        "IL10->IL10RA",
        "IL10->IL10RB",
    ],

    "Fate: B-lineage NOTCH2": [
        "JAG1->NOTCH2",
        "JAG2->NOTCH2",
        "DLL1->NOTCH2",
        "DLL4->NOTCH2",
    ],

    "Survival: memory T IL-15": [
        "IL15->IL15RA",
        "IL15->IL2RB",
    ],

    "Co-signalling: T coinhibitory": [
        "PVR->TIGIT",
        "NECTIN2->TIGIT",
        "NECTIN3->TIGIT",
        "PVR->CD96",
        "CD86->CTLA4",
    ],

    "Co-signalling: T activating": [
        "CD226->NECTIN2",
        "CD86->CD28",
        "ICOSLG->ICOS",
        "CD70->CD27",
    ],

    "Myeloid: ITIM-coupled inhibitory": [
        "HLA-F->LILRB1",
        "HLA-F->LILRB2",
        "LAIR1->LILRB4",
        "CD47->SIRPA",
        "LGALS9->HAVCR2",
    ],

    "Myeloid: efferocytic": [
        "GAS6->MERTK",
        "GAS6->AXL",
        "LGALS3->MERTK",
        "APOE->TREM2",
        "APOE->TYROBP",
        "CLU->TREM2",
        "CLU->TYROBP",
    ],

    "Myeloid: complement and CSF1R": [
        "C3->C3AR1",
        "C3->C5AR2",
        "C3->ITGAM",
        "C3->ITGB2",
        "CSF1->CSF1R",
        "IL34->CSF1R",
    ],

    "Lost: thymocyte- and DC1-derived help": [
        "CD40LG->CD40",
        "CD40LG->ITGB1",
        "CD40LG->ITGA5",
        "LTA->TNFRSF14",
        "LTA->TNFRSF1B",
        "BTLA->TNFRSF14",
        "LTB->LTBR",
        "CXCL10->CXCR3",
        "LGALS9->PTPRC",
    ],
}}

all_pairs = [pair for module in MODULES.values() for pairs in module.values() for pair in pairs]

In [ ]:
contrasts = [('adult', 'paed'), ('adult', 'infant'), ('paed', 'infant')]

combined_niche_ranksums = [pd.read_csv(f'{data_path}/analyses/cellComm/recirc/combined_niche_{target_age_group[1]}_vs_{target_age_group[0]}_ranksums.csv', index_col = 0) for target_age_group in contrasts]
nrows = [df.shape[0] for df in combined_niche_ranksums]

combined_niche_ranksums = pd.concat(combined_niche_ranksums, axis = 0)
combined_niche_ranksums['contrasts'] = np.repeat([f'{c[0]}_vs_{c[1]}' for c in contrasts], nrows)

combined_niche_ranksums

In [ ]:
combined_niche_ranksums_filtred = combined_niche_ranksums.loc[(combined_niche_ranksums['interaction'].isin(all_pairs))]
combined_niche_ranksums_filtred

In [ ]:
combined_niche_ranksums_filtred.to_csv(f'{data_path}/analyses/cellComm/recirc/combined_niche_ranksums_filtered.csv', index=False)

In [ ]:
combined_niche_ranksums_filtred.statistic.describe()

In [ ]:
from plotting.colors import thyAgeing_colors
from matplotlib.backends.backend_pdf import PdfPages

# Keep only adhesion-related interactions if available
for module_name in MODULES.keys():
    module_pairs = [pair for pairs in MODULES[module_name].values() for pair in pairs]
    df_all = combined_niche_ranksums_filtred.loc[combined_niche_ranksums_filtred['interaction'].isin(module_pairs)].copy()

    pdf_path = f'{plots_path}/cellComm/recirc/combined_niches_{module_name.replace(" ", "_")}_ranksums_heatmaps.pdf'

    contrast_order = [c for c in ['adult_vs_infant', 'paed_vs_infant', 'adult_vs_paed'] if c in df_all['contrasts'].unique()]
    contrast_order += [c for c in df_all['contrasts'].dropna().unique() if c not in contrast_order]

    with PdfPages(pdf_path) as pdf:
        for contrast in contrast_order:
            df_sub = df_all[df_all['contrasts'] == contrast].copy()

            df_sub = df_sub.drop_duplicates(subset=['interaction', 'cells'])
            heat = df_sub.pivot(index='interaction', columns='cells', values='statistic')
            heat_anno = df_sub.pivot(index='interaction', columns='cells', values='padj').applymap(lambda x: '*' if x < 0.05 else '')

            order = heat.abs().max(axis=1).sort_values(ascending=False).index
            heat = heat.reindex(order)
            heat_anno = heat_anno.reindex(order)

            absmax = 4.0
            vmin, vmax = -absmax, absmax

            fig, ax = plt.subplots(figsize=calc_figsize(width=heat.shape[1] * 2.5 + 20, height=heat.shape[0] * 2.5 + 20))
            sns.heatmap(
                heat,
                cmap=sns.blend_palette([thyAgeing_colors['teal'], 'white', thyAgeing_colors['orange']], as_cmap=True),
                center=0,
                vmin=vmin,
                vmax=vmax,
                annot=heat_anno,
                fmt='',
                annot_kws={'fontsize': 8},
                cbar_kws={'label': 'ranksums statistic'},
                linewidths=0.0,
                xticklabels=1,   # force all x tick labels
                yticklabels=1,   # force all y tick labels
                ax=ax
            )

            # Explicitly set all ticks to prevent auto-skipping
            ax.set_xticks(np.arange(heat.shape[1]) + 0.5)
            ax.set_xticklabels(heat.columns, rotation=90)
            ax.set_yticks(np.arange(heat.shape[0]) + 0.5)
            ax.set_yticklabels(heat.index, rotation=0)

            ax.set_xlabel('cells')
            ax.set_ylabel(f'{module_name.title()} interaction')
            ax.set_title(f'{contrast} — statistic heatmap')
            fig.tight_layout()
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)

    print(f'Saved: {pdf_path}')

In [ ]:
contrasts = [('adult', 'paed'), ('adult', 'infant')]

shared_niche_ranksums = [pd.read_csv(f'{data_path}/analyses/cellComm/recirc/shared_niche_{target_age_group[1]}_vs_{target_age_group[0]}_ranksums.csv', index_col = 0) for target_age_group in contrasts]
nrows = [df.shape[0] for df in shared_niche_ranksums]

shared_niche_ranksums = pd.concat(shared_niche_ranksums, axis = 0)
shared_niche_ranksums['contrasts'] = np.repeat([f'{c[0]}_vs_{c[1]}' for c in contrasts], nrows)
shared_niche_ranksums_filtred = shared_niche_ranksums.loc[shared_niche_ranksums['interaction'].isin(all_pairs)]
shared_niche_ranksums_filtred

In [ ]:
shared_niche_ranksums_filtred.to_csv(f'{data_path}/analyses/cellComm/recirc/shared_niche_ranksums_filtered.csv', index=False)

In [ ]:
from plotting.colors import thyAgeing_colors
from matplotlib.backends.backend_pdf import PdfPages

# Keep only adhesion-related interactions if available
for module_name in MODULES.keys():
    module_pairs = [pair for pairs in MODULES[module_name].values() for pair in pairs]
    df_all = shared_niche_ranksums_filtred.loc[shared_niche_ranksums_filtred['interaction'].isin(module_pairs)].copy()

    pdf_path = f'{plots_path}/cellComm/recirc/shared_niches_{module_name.replace(" ", "_")}_ranksums_heatmaps.pdf'

    contrast_order = [c for c in ['adult_vs_infant', 'adult_vs_paed'] if c in df_all['contrasts'].unique()]
    contrast_order += [c for c in df_all['contrasts'].dropna().unique() if c not in contrast_order]

    with PdfPages(pdf_path) as pdf:
        for contrast in contrast_order:
            df_sub = df_all[df_all['contrasts'] == contrast].copy()

            df_sub = df_sub.drop_duplicates(subset=['interaction', 'cells'])
            heat = df_sub.pivot(index='interaction', columns='cells', values='statistic')
            heat_anno = df_sub.pivot(index='interaction', columns='cells', values='padj').applymap(lambda x: '*' if x < 0.05 else '')

            order = heat.abs().max(axis=1).sort_values(ascending=False).index
            heat = heat.reindex(order)
            heat_anno = heat_anno.reindex(order)

            absmax = 4.0
            vmin, vmax = -absmax, absmax

            fig, ax = plt.subplots(figsize=calc_figsize(width=heat.shape[1] * 2.5 + 20, height=heat.shape[0] * 2.5 + 20))
            sns.heatmap(
                heat,
                cmap=sns.blend_palette([thyAgeing_colors['teal'], 'white', thyAgeing_colors['orange']], as_cmap=True),
                center=0,
                vmin=vmin,
                vmax=vmax,
                annot=heat_anno,
                fmt='',
                annot_kws={'fontsize': 8},
                cbar_kws={'label': 'ranksums statistic'},
                linewidths=0.0,
                xticklabels=1,   # force all x tick labels
                yticklabels=1,   # force all y tick labels
                ax=ax
            )

            # Explicitly set all ticks to prevent auto-skipping
            ax.set_xticks(np.arange(heat.shape[1]) + 0.5)
            ax.set_xticklabels(heat.columns, rotation=90)
            ax.set_yticks(np.arange(heat.shape[0]) + 0.5)
            ax.set_yticklabels(heat.index, rotation=0)

            ax.set_xlabel('cells')
            ax.set_ylabel(f'{module_name.title()} interaction')
            ax.set_title(f'{contrast} — statistic heatmap')
            fig.tight_layout()
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)

    print(f'Saved: {pdf_path}')

In [ ]:
from plotting.colors import thyAgeing_colors
from matplotlib.backends.backend_pdf import PdfPages

# Keep only adhesion-related interactions if available
df_all = shared_niche_ranksums_filtred.copy()
sel = df_all['classification'].str.contains('hesion') #Chemokines|Interleukin|Tumor necrosis factor|Transforming growth factor|Vascular endothelial growth factor
sel = sel.fillna(False).astype(bool)
df_all = df_all.loc[sel].copy()

pdf_path = f'{plots_path}/cellComm/recirc/shared_niches_adhesion_ranksums_heatmaps.pdf'

contrast_order = [c for c in ['infant_vs_adult', 'infant_vs_paed', 'paed_vs_adult'] if c in df_all['contrasts'].unique()]
contrast_order += [c for c in df_all['contrasts'].dropna().unique() if c not in contrast_order]

with PdfPages(pdf_path) as pdf:
    for contrast in contrast_order:
        df_sub = df_all[df_all['contrasts'] == contrast].copy()

        if df_sub.empty:
            fig, ax = plt.subplots(figsize=calc_figsize(width=60, height=40))
            ax.axis('off')
            ax.text(0.5, 0.5, f'No data available for {contrast}', ha='center', va='center')
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)
            continue

        df_sub = df_sub.drop_duplicates(subset=['interaction', 'cells'])
        heat = df_sub.pivot(index='interaction', columns='cells', values='statistic')
        heat_anno = df_sub.pivot(index='interaction', columns='cells', values='padj').applymap(lambda x: '*' if x < 0.05 else '')

        order = heat.abs().max(axis=1).sort_values(ascending=False).index
        heat = heat.reindex(order)
        heat_anno = heat_anno.reindex(order)

        absmax = np.nanmax(np.abs(heat.values))
        if not np.isfinite(absmax) or absmax == 0:
            absmax = 1.0
        vmin, vmax = -absmax, absmax

        fig, ax = plt.subplots(figsize=calc_figsize(width=60, height=max(60, 6 * len(heat) / 2)))
        sns.heatmap(
            heat,
            cmap=sns.blend_palette([thyAgeing_colors['teal'], 'white', thyAgeing_colors['orange']], as_cmap=True),
            center=0,
            vmin=vmin,
            vmax=vmax,
            annot=heat_anno,
            fmt='',
            annot_kws={'fontsize': 8},
            cbar_kws={'label': 'ranksums statistic'},
            linewidths=0.0,
            xticklabels=1,   # force all x tick labels
            yticklabels=1,   # force all y tick labels
            ax=ax
        )

        # Explicitly set all ticks to prevent auto-skipping
        ax.set_xticks(np.arange(heat.shape[1]) + 0.5)
        ax.set_xticklabels(heat.columns, rotation=90)
        ax.set_yticks(np.arange(heat.shape[0]) + 0.5)
        ax.set_yticklabels(heat.index, rotation=0)

        ax.set_xlabel('cells')
        ax.set_ylabel('Adhesion interaction')
        ax.set_title(f'{contrast} — statistic heatmap')
        fig.tight_layout()
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

print(f'Saved: {pdf_path}')